# Context-Aware Notification Engine (CANE)
## Deep Q-Network (DQN)

This notebook implements a **Deep Q-Network (DQN)** agent for the Context-Aware Notification Engine (CANE).

DQN models the notification pacing problem as a full Markov Decision Process (MDP). It approximates the optimal action-value function $Q^*(s, a)$ using a deep neural network, trained via Bellman error minimization with experience replay and a stationary target network:

$$\mathcal{L}(\theta) = \mathbb{E}_{(s, a, r, s', d) \sim \mathcal{D}} \left[ \left( r + \gamma \cdot \max_{a' \in \mathcal{A}} Q_{\text{target}}(s', a') \cdot (1 - d) - Q_{\text{online}}(s, a; \theta) \right)^2 \right]$$

**Notebook Structure**

| Section | Contents |
|---|---|
| 1 | Setup & Dependencies |
| 2 | Agent Interface |
| 3 | Environment Dynamics (`CANEEnv`) |
| 4 | Baselines & Evaluation Protocol |
| 5 | Environment Sanity Checks & Plotting Utilities |
| 6 | Deep Q-Network Architecture & Agent Implementation |
| 7 | Correctness Verification Tests |
| 8 | Registry & Execution Driver |
| 9 | Learning Curves & Optimisation Diagnostics |
| 10 | Decision Distribution & Policy Behaviour |
| 11 | Hyperparameter Sensitivity Analysis |
| 12 | Statistical Robustness & Model Persistence |


---

## 1. Setup & Dependencies

Import required libraries, define action spaces, observation feature layouts, and initialize execution parameters.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from abc import ABC, abstractmethod
import torch
import torch.nn as nn

# --- Action Space -----------------------------------------------------------
HOLD = 0        # Stay silent; fatigue decays
ENGAGE = 1      # Engagement notification
INCENTIVE = 2   # Incentive / promo notification

N_ACTIONS = 3
ACTION_NAMES = {HOLD: "Hold", ENGAGE: "Engage", INCENTIVE: "Incentive"}

# --- Observation Feature Indices --------------------------------------------
IDX_HOUR = 0        # Raw hour (0-23)
IDX_DAY = 1         # Raw day of week (0-6)
IDX_ACTIVE = 2      # In-app activity flag (0 or 1)
IDX_FATIGUE = 3     # Accumulated fatigue [0, 1]
IDX_RECENCY = 4     # Normalised time since last send [0, 1]

N_BLOCKS = 8        # 3-hour blocks per day
BLOCK_HOURS = 24 // N_BLOCKS

IDX_ACT_RATE = 5    # Block activity rate (5..12)
IDX_CLICK_RATE = 13 # Block click rate (13..20)
IDX_TYPE_CR = 21    # Click rate per message type (21..22)
IDX_SINCE_CLICK = 23# Hours since last click

STATE_DIM_BASE = 5
STATE_DIM = 24

BELIEF_PRIOR = (1.0, 4.0)
USE_BELIEF = True

def agent_state_dim():
    return STATE_DIM if USE_BELIEF else STATE_DIM_BASE

DEVICE = "cpu"
torch.set_num_threads(1)

print("Actions:", ACTION_NAMES)
print("State dim:", STATE_DIM, "| agent sees:", agent_state_dim())
print("torch version:", torch.__version__, "| device:", DEVICE)

---

## 2. Agent Interface

Abstract base class defining the shared execution contract for all agents.

In [ ]:
class Agent(ABC):
    """Abstract base class for all CANE agents."""

    @property
    def name(self):
        return self.__class__.__name__

    @abstractmethod
    def act(self, state, greedy=False):
        """Select an action given an observation state."""
        raise NotImplementedError

    @abstractmethod
    def update(self, state, action, reward, next_state, done, aux):
        """Update policy / value function from a transition."""
        raise NotImplementedError

    def reset(self):
        """Reset agent state at episode boundaries (default no-op)."""
        pass

print("Agent interface defined.")

---

## 3. Environment Dynamics

Defines user archetypes, click/fatigue/churn dynamics, and the `CANEEnv` environment.

In [ ]:
# --- Environment Parameters --------------------------------------------------
ARCHETYPE_W = {
    "OfficeWorker":     {ENGAGE: 1.15, INCENTIVE: 1.05},
    "NightOwlStudent":  {ENGAGE: 0.90, INCENTIVE: 1.60},
    "NightShiftWorker": {ENGAGE: 1.00, INCENTIVE: 1.25},
    "NormalStudent":    {ENGAGE: 0.90, INCENTIVE: 1.55},
    "Housewife":        {ENGAGE: 1.05, INCENTIVE: 1.45},
}

CANE_CONFIG = dict(
    steps_per_episode=168,
    lam=0.90,
    kappa={HOLD: 0.0, ENGAGE: 0.12, INCENTIVE: 0.18},
    mu=0.80,
    w_arch=ARCHETYPE_W,
    churn_threshold=0.70,
    gamma_0=-7.0,
    gamma_1=4.0,
    R_click=10.0,
    W_send={HOLD: 0.0, ENGAGE: 1.0, INCENTIVE: 2.5},
    W_fat=2.0,
    R_churn=30.0,
    beta=1.15,
    streak_mode="marginal",
    streak_cap=5.0,
    streak_lapse_hours=36,
    active_scale=0.6,
)

ARCHETYPES = ["OfficeWorker", "NightOwlStudent", "NightShiftWorker",
              "NormalStudent", "Housewife"]

def _bump(hours, centre, width, amp):
    delta = (hours - centre + 12.0) % 24.0 - 12.0
    return amp * np.exp(-0.5 * (delta / width) ** 2)

def archetype_curve(archetype, hours=None):
    h = np.arange(24.0) if hours is None else np.asarray(hours, dtype=float)
    if archetype == "OfficeWorker":
        c = _bump(h, 7.5, 1.0, 0.42) + _bump(h, 12.5, 1.1, 0.30) + _bump(h, 19.5, 2.0, 0.50)
    elif archetype == "NightOwlStudent":
        c = _bump(h, 0.0, 2.2, 0.32) + _bump(h, 22.0, 1.6, 0.26) + _bump(h, 16.0, 2.0, 0.10)
    elif archetype == "NightShiftWorker":
        c = _bump(h, 16.5, 1.6, 0.45) + _bump(h, 6.5, 1.5, 0.40) + _bump(h, 2.0, 1.8, 0.18)
    elif archetype == "NormalStudent":
        c = _bump(h, 16.0, 1.6, 0.45) + _bump(h, 21.0, 1.5, 0.42) + _bump(h, 7.0, 1.0, 0.20)
    elif archetype == "Housewife":
        c = _bump(h, 10.0, 2.4, 0.42) + _bump(h, 14.5, 2.4, 0.40) + _bump(h, 20.5, 1.6, 0.22)
    else:
        raise ValueError(f"unknown archetype: {archetype}")
    return np.clip(c, 0.0, 1.0)

ARCHETYPE_TABLE = {a: archetype_curve(a) for a in ARCHETYPES}
WEEKEND_FLATTEN = {"OfficeWorker": 0.55, "NormalStudent": 0.50,
                   "NightShiftWorker": 0.30, "NightOwlStudent": 0.15,
                   "Housewife": 0.05}

print(f"{len(ARCHETYPES)} archetypes defined:", ", ".join(ARCHETYPES))

In [ ]:
class CANEEnv:
    """Context-Aware Notification Engine simulator environment."""

    def __init__(self, config=None, archetype=None, seed=0):
        self.cfg = {**CANE_CONFIG, **(config or {})}
        self.fixed_archetype = archetype
        self.rng = np.random.default_rng(seed)
        self.reset()

    def _p0(self):
        base = ARCHETYPE_TABLE[self.archetype][self.hour]
        if self.day >= 5:
            f = WEEKEND_FLATTEN[self.archetype]
            base = (1.0 - f) * base + f * ARCHETYPE_TABLE[self.archetype].mean()
        return float(base)

    def _observe(self):
        s = np.zeros(STATE_DIM, dtype=np.float32)
        s[IDX_HOUR] = self.hour
        s[IDX_DAY] = self.day
        s[IDX_ACTIVE] = self.active
        s[IDX_FATIGUE] = self.fatigue
        s[IDX_RECENCY] = self.recency
        a0, b0 = BELIEF_PRIOR
        s[IDX_ACT_RATE:IDX_ACT_RATE + N_BLOCKS] = (self.blk_act_hits + a0) / (self.blk_act_obs + a0 + b0)
        s[IDX_CLICK_RATE:IDX_CLICK_RATE + N_BLOCKS] = (self.blk_clicks + a0) / (self.blk_sends + a0 + b0)
        s[IDX_TYPE_CR] = (self.type_clicks[ENGAGE] + a0) / (self.type_sends[ENGAGE] + a0 + b0)
        s[IDX_TYPE_CR + 1] = (self.type_clicks[INCENTIVE] + a0) / (self.type_sends[INCENTIVE] + a0 + b0)
        s[IDX_SINCE_CLICK] = min(self.hours_since_click / 48.0, 1.0)
        return s

    def _roll_activity(self):
        p = min(1.0, self.cfg["active_scale"] * self._p0())
        self.active = int(self.rng.random() < p)
        b = self.hour // BLOCK_HOURS
        self.blk_act_obs[b] += 1
        self.blk_act_hits[b] += self.active

    def reset(self, seed=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        self.archetype = self.fixed_archetype or ARCHETYPES[self.rng.integers(len(ARCHETYPES))]
        self.t = 0
        self.hour = int(self.rng.integers(24))
        self.day = int(self.rng.integers(7))
        self.fatigue = 0.0
        self.recency = 1.0
        self.hours_since_send = 24
        self.streak = 0
        self.hours_since_click = 0
        self.opted_out = False
        self.blk_act_obs = np.zeros(N_BLOCKS)
        self.blk_act_hits = np.zeros(N_BLOCKS)
        self.blk_sends = np.zeros(N_BLOCKS)
        self.blk_clicks = np.zeros(N_BLOCKS)
        self.type_sends = np.zeros(N_ACTIONS)
        self.type_clicks = np.zeros(N_ACTIONS)
        self._roll_activity()
        return self._observe(), {"archetype": self.archetype}

    def step(self, action):
        cfg = self.cfg
        action = int(action)
        sent = action != HOLD
        f_now = self.fatigue

        clicked = False
        if sent and not self.active:
            w_a = cfg["w_arch"][self.archetype][action]
            p_click = np.clip(self._p0() * (1.0 - cfg["mu"] * f_now) * w_a, 0.0, 1.0)
            clicked = bool(self.rng.random() < p_click)

        blk = self.hour // BLOCK_HOURS
        if sent:
            self.blk_sends[blk] += 1
            self.blk_clicks[blk] += clicked
            self.type_sends[action] += 1
            self.type_clicks[action] += clicked

        if clicked:
            self.streak += 1
            self.hours_since_click = 0
        else:
            self.hours_since_click += 1
            if self.hours_since_click > cfg["streak_lapse_hours"]:
                self.streak = 0

        streak_bonus = 0.0
        if clicked and self.streak > 0:
            if cfg["streak_mode"] == "cumulative":
                streak_bonus = sum(cfg["beta"] ** k for k in range(1, self.streak + 1))
            else:
                streak_bonus = cfg["beta"] ** self.streak
            if cfg["streak_cap"] is not None:
                streak_bonus = min(streak_bonus, cfg["streak_cap"])

        self.fatigue = float(np.clip(cfg["lam"] * f_now + cfg["kappa"][action], 0.0, 1.0))

        if self.fatigue > cfg["churn_threshold"]:
            hazard = 1.0 / (1.0 + np.exp(-(cfg["gamma_0"] + cfg["gamma_1"] * self.fatigue)))
            self.opted_out = bool(self.rng.random() < hazard)

        reward = (cfg["R_click"] * clicked
                  - cfg["W_send"][action]
                  - cfg["W_fat"] * f_now
                  - cfg["R_churn"] * self.opted_out
                  + streak_bonus)

        self.t += 1
        self.hour = (self.hour + 1) % 24
        if self.hour == 0:
            self.day = (self.day + 1) % 7

        self.hours_since_send = 0 if sent else self.hours_since_send + 1
        self.recency = float(min(self.hours_since_send / 24.0, 1.0))
        self._roll_activity()

        terminated = self.opted_out
        truncated = self.t >= cfg["steps_per_episode"]

        info = {
            "archetype": self.archetype,
            "clicked": clicked, "sent": sent,
            "fatigue": self.fatigue, "fatigue_pre": f_now,
            "opted_out": self.opted_out, "streak": self.streak,
            "streak_bonus": streak_bonus,
            "hour": int((self.hour - 1) % 24), "active_pre": self.active,
        }
        return self._observe(), float(reward), terminated, truncated, info

print("CANEEnv defined.")

---

## 4. Baselines & Evaluation Protocol

Provides benchmark agents (`FixedScheduleAgent`, `RandomAgent`) and evaluation harness functions.

In [ ]:
class FixedScheduleAgent(Agent):
    """Sends an Engagement Nudge at a fixed hour every day."""
    def __init__(self, hour=18, action=ENGAGE):
        self.hour, self.action = hour, action

    @property
    def name(self):
        return f"Fixed-{self.hour:02d}:00"

    def act(self, state, greedy=False):
        return (self.action if int(state[IDX_HOUR]) == self.hour else HOLD), {}

    def update(self, *args, **kwargs):
        return {}


class RandomAgent(Agent):
    """Selects actions uniformly at random."""
    def __init__(self, seed=0):
        self.rng = np.random.default_rng(seed)

    @property
    def name(self):
        return "Random"

    def act(self, state, greedy=False):
        return int(self.rng.integers(N_ACTIONS)), {}

    def update(self, *args, **kwargs):
        return {}


def run_episodes(agent, env, n_episodes=None, seeds=None, learn=True,
                 greedy=False, collect_log=False):
    """Runs agent in env over specified episodes and returns evaluation metrics."""
    if seeds is None:
        seeds = [None] * int(n_episodes)

    ep_rewards, ep_sends, ep_clicks, ep_optouts, ep_lengths = [], [], [], [], []
    hour_actions = np.zeros((24, N_ACTIONS), dtype=int)
    log = []

    for ep, sd in enumerate(seeds):
        state, info = env.reset(seed=sd)
        agent.reset()
        total_r = sends = clicks = 0.0

        while True:
            action, aux = agent.act(state, greedy=greedy)
            next_state, reward, terminated, truncated, info = env.step(action)

            if learn:
                agent.update(state, action, reward, next_state, terminated, aux)

            total_r += reward
            sends += info["sent"]
            clicks += info["clicked"]
            hour_actions[info["hour"], action] += 1

            if collect_log:
                log.append({"episode": ep, "step": env.t, "hour": info["hour"],
                            "archetype": info["archetype"], "action": action,
                            "clicked": info["clicked"], "sent": info["sent"],
                            "fatigue": info["fatigue"], "reward": reward,
                            "streak": info["streak"], "opted_out": info["opted_out"]})

            state = next_state
            if terminated or truncated:
                ep_optouts.append(float(terminated))
                break

        ep_rewards.append(total_r)
        ep_sends.append(sends)
        ep_clicks.append(clicks)
        ep_lengths.append(env.t)

    total_sends = float(np.sum(ep_sends))
    metrics = {
        "agent": agent.name,
        "reward_mean": float(np.mean(ep_rewards)),
        "reward_std": float(np.std(ep_rewards)),
        "ctr": float(np.sum(ep_clicks) / total_sends) if total_sends > 0 else 0.0,
        "sends_per_episode": float(np.mean(ep_sends)),
        "clicks_per_episode": float(np.mean(ep_clicks)),
        "optout_rate": float(np.mean(ep_optouts)),
        "episode_length": float(np.mean(ep_lengths)),
    }
    return metrics, (log if collect_log else None), np.array(ep_rewards), hour_actions


QUICK = False

EVAL_SEEDS = list(range(900_000, 900_050 if QUICK else 900_200))
TRAIN_EPISODES = 150 if QUICK else 600
N_SEEDS = 2 if QUICK else 5

print(f"Eval protocol: {len(EVAL_SEEDS)} reserved episodes; "
      f"training budget {TRAIN_EPISODES} episodes; {N_SEEDS} seeds."
      + (" [QUICK]" if QUICK else ""))

In [ ]:
PROBE_SEEDS = list(range(950_000, 950_020))
SNAPSHOT_ARCHETYPES = ("OfficeWorker", "NightOwlStudent")

def policy_snapshot(agent, archetypes=SNAPSHOT_ARCHETYPES, seeds=PROBE_SEEDS):
    """Evaluate frozen greedy policy across probe seeds."""
    out = {}
    for arch in archetypes:
        env = CANEEnv(seed=8000, archetype=arch)
        m, _, _, hours = run_episodes(agent, env, seeds=seeds, learn=False, greedy=True)
        out[arch] = {"hour_actions": hours, "reward": m["reward_mean"],
                     "ctr": m["ctr"], "sends": m["sends_per_episode"],
                     "optout": m["optout_rate"]}
    return out

def train_and_evaluate(make_agent, n_seeds=None, train_episodes=TRAIN_EPISODES,
                       archetype=None, collect_hours=False,
                       snapshot_every=None, snapshot_seed=0,
                       snapshot_archetypes=SNAPSHOT_ARCHETYPES):
    """Train an agent across multiple random seeds and evaluate on held-out seeds."""
    n_seeds = N_SEEDS if n_seeds is None else n_seeds
    per_seed, hour_stack, snapshots = [], [], []

    for seed in range(n_seeds):
        agent = make_agent(seed)
        train_env = CANEEnv(seed=1000 + seed, archetype=archetype)

        if snapshot_every and seed == snapshot_seed:
            snapshots.append((0, policy_snapshot(agent, snapshot_archetypes)))
            done = 0
            while done < train_episodes:
                chunk = min(snapshot_every, train_episodes - done)
                run_episodes(agent, train_env, n_episodes=chunk, learn=True, greedy=False)
                done += chunk
                snapshots.append((done, policy_snapshot(agent, snapshot_archetypes)))
        else:
            run_episodes(agent, train_env, n_episodes=train_episodes, learn=True, greedy=False)

        eval_env = CANEEnv(seed=7000 + seed, archetype=archetype)
        m, _, rewards, hours = run_episodes(agent, eval_env, seeds=EVAL_SEEDS, learn=False, greedy=True)
        per_seed.append(m)
        hour_stack.append(hours)

    rewards = np.array([m["reward_mean"] for m in per_seed])
    summary = {
        "agent": per_seed[0]["agent"],
        "reward_mean": rewards.mean(),
        "reward_std": rewards.std(ddof=0),
        "reward_sem_std": rewards.std(ddof=1) if n_seeds > 1 else 0.0,
        "ctr": np.mean([m["ctr"] for m in per_seed]),
        "sends_per_episode": np.mean([m["sends_per_episode"] for m in per_seed]),
        "optout_rate": np.mean([m["optout_rate"] for m in per_seed]),
        "per_seed_rewards": rewards,
    }
    if collect_hours:
        summary["hour_actions"] = np.sum(hour_stack, axis=0)
    if snapshots:
        summary["snapshots"] = snapshots
    return summary

def evaluate_baseline(make_agent, archetype=None):
    agent = make_agent(0)
    env = CANEEnv(seed=7000, archetype=archetype)
    m, _, rewards, hours = run_episodes(agent, env, seeds=EVAL_SEEDS, learn=False, greedy=True)
    return {"agent": m["agent"], "reward_mean": m["reward_mean"],
            "reward_std": m["reward_std"], "reward_sem_std": 0.0,
            "ctr": m["ctr"], "sends_per_episode": m["sends_per_episode"],
            "optout_rate": m["optout_rate"],
            "per_seed_rewards": np.array([m["reward_mean"]]),
            "hour_actions": hours}

---

## 5. Environment Sanity Checks & Exploratory Data Analysis (EDA)

Validates environment dynamics and visualises baseline responsiveness curves, fatigue dynamics, click surfaces, and ground-truth policy reference maps.

In [ ]:
# --- 5.1 Environment Sanity Checks -----------------------------------------------
env = CANEEnv(seed=0)

e = CANEEnv(archetype="Housewife", seed=1)
e.reset(seed=1)
f_trace, expected = [], 0.0
for _ in range(30):
    _, _, term, trunc, info = e.step(ENGAGE)
    expected = min(0.9 * expected + CANE_CONFIG["kappa"][ENGAGE], 1.0)
    f_trace.append((info["fatigue"], expected))
    if term or trunc:
        break
assert all(abs(a - b) < 1e-9 for a, b in f_trace)

e.reset(seed=2)
for _ in range(5):
    e.step(ENGAGE)
f_high = e.fatigue
for _ in range(10):
    e.step(HOLD)
assert e.fatigue < f_high

class _AlwaysHold(Agent):
    def act(self, s, greedy=False): return HOLD, {}
    def update(self, *a, **k): return {}

m_hold, _, _, _ = run_episodes(_AlwaysHold(), CANEEnv(seed=3), 30, learn=False, greedy=True)
assert m_hold["optout_rate"] == 0.0

class _AlwaysSend(Agent):
    def act(self, s, greedy=False): return ENGAGE, {}
    def update(self, *a, **k): return {}

m_spam, _, _, _ = run_episodes(_AlwaysSend(), CANEEnv(seed=4), 30, learn=False, greedy=True)
assert m_spam["optout_rate"] > 0.8

print("Environment sanity checks passed.")

In [ ]:
# --- Plot G1: Receptive Windows by Archetype ---
from pathlib import Path

FOCUS = ["OfficeWorker", "NightOwlStudent"]
ARCH_COLOUR = {"OfficeWorker": "#2f6fb5", "NightOwlStudent": "#c4453c"}

FIGDIR = Path("figures")
FIGDIR.mkdir(exist_ok=True)

def save(fig, tag):
    fig.savefig(FIGDIR / f"{tag}.png", dpi=150, bbox_inches="tight")

def weekend_curve(archetype):
    base = ARCHETYPE_TABLE[archetype]
    f = WEEKEND_FLATTEN[archetype]
    return (1.0 - f) * base + f * base.mean()

hours = np.arange(24)
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4), sharey=True)
for ax, arch in zip(axes, FOCUS):
    ax.fill_between(hours, ARCHETYPE_TABLE[arch], color=ARCH_COLOUR[arch], alpha=0.15)
    ax.plot(hours, ARCHETYPE_TABLE[arch], color=ARCH_COLOUR[arch], lw=2.0, label="weekday")
    ax.plot(hours, weekend_curve(arch), color=ARCH_COLOUR[arch], lw=1.3, ls="--", label="weekend")
    ax.set_title(arch, fontsize=10)
    ax.set_xlabel("hour of day")
    ax.set_xticks(range(0, 24, 3))
    ax.legend(frameon=False, fontsize=8)
axes[0].set_ylabel(r"$p_0$  click propensity")
fig.suptitle("G1   Receptive windows by archetype", fontsize=11)
fig.tight_layout()
save(fig, "G1_archetype_curves")
plt.show()

In [ ]:
# --- Plot G2: Fatigue Dynamics ---
lam = CANE_CONFIG["lam"]
kap = CANE_CONFIG["kappa"]
thr = CANE_CONFIG["churn_threshold"]
pattern = np.array([1, 1, 1, 1, 1, 1] + [0] * 18)

fig, ax = plt.subplots(figsize=(9, 3.0))
ax.fill_between(range(len(pattern)), 0, 1, where=pattern.astype(bool),
                color="#999999", alpha=0.12, step="post")
for act, col in [(ENGAGE, "#2f6fb5"), (INCENTIVE, "#c4453c")]:
    F, trace = 0.0, []
    for a in pattern:
        F = float(np.clip(lam * F + (kap[act] if a else 0.0), 0.0, 1.0))
        trace.append(F)
    ax.step(range(len(trace)), trace, where="post", lw=1.8, color=col,
            label=f"{ACTION_NAMES[act]}  (kappa={kap[act]})")

ax.axhline(thr, color="#666666", ls=":", lw=1.4)
ax.text(len(pattern) - 0.5, thr + 0.015, f"churn threshold {thr}",
        ha="right", va="bottom", fontsize=8, color="#666666")
ax.set_xlabel("hours")
ax.set_ylabel("fatigue $F$")
ax.set_ylim(0, 1)
ax.legend(frameon=False, fontsize=8, loc="upper right")
ax.set_title("G2   Six hourly sends (shaded), then silence", fontsize=11)
fig.tight_layout()
save(fig, "G2_fatigue_dynamics")
plt.show()

In [ ]:
# --- Plot G3: Click Probability Surface ---
mu = CANE_CONFIG["mu"]

def break_even(archetype, action):
    cfg = CANE_CONFIG
    debt = cfg["W_fat"] * cfg["kappa"][action] / (1.0 - cfg["lam"])
    return (debt + cfg["W_send"][action]) / (cfg["R_click"] * cfg["w_arch"][archetype][action])

F_grid = np.linspace(0.0, 1.0, 101)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for ax, arch in zip(axes, FOCUS):
    P = np.clip(np.outer(1.0 - mu * F_grid, ARCHETYPE_TABLE[arch])
                * CANE_CONFIG["w_arch"][arch][ENGAGE], 0.0, 1.0)
    im = ax.imshow(P, origin="lower", aspect="auto", cmap="magma",
                   extent=[0, 23, 0, 1], vmin=0, vmax=0.6)
    cs = ax.contour(np.arange(24), F_grid, P, levels=[break_even(arch, ENGAGE)],
                    colors="w", linewidths=1.5)
    ax.clabel(cs, fmt={break_even(arch, ENGAGE): "break-even"}, fontsize=7)
    ax.set_title(f"{arch}   (Engage break-even {break_even(arch, ENGAGE):.2f})", fontsize=9)
    ax.set_xlabel("hour of day")
    ax.set_xticks(range(0, 24, 3))
axes[0].set_ylabel("fatigue $F$")
fig.colorbar(im, ax=axes, label="P(click | Engage)")
fig.suptitle("G3   Engage click probability surface", fontsize=11)
save(fig, "G3_click_surface")
plt.show()

In [ ]:
# --- Plot G4: Ground-Truth Policy Reference ---
def net_value(archetype, action, hour, F):
    cfg = CANE_CONFIG
    p = min(ARCHETYPE_TABLE[archetype][hour] * (1.0 - cfg["mu"] * F)
            * cfg["w_arch"][archetype][action], 1.0)
    debt = cfg["W_fat"] * cfg["kappa"][action] / (1.0 - cfg["lam"])
    return p * cfg["R_click"] - cfg["W_send"][action] - debt

F_GRID = np.linspace(0.0, 1.0, 101)

GROUND_TRUTH = {}
for arch in ARCHETYPES:
    M = np.zeros((len(F_GRID), 24), dtype=int)
    for j, F in enumerate(F_GRID):
        for h in range(24):
            v = {a: net_value(arch, a, h, F) for a in (ENGAGE, INCENTIVE)}
            best = max(v, key=v.get)
            M[j, h] = best if v[best] > 0 else HOLD
    GROUND_TRUTH[arch] = M

from matplotlib.colors import ListedColormap
ACTION_COLOUR = ["#e8e8e8", "#2f6fb5", "#c4453c"]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for ax, arch in zip(axes, FOCUS):
    ax.imshow(GROUND_TRUTH[arch], origin="lower", aspect="auto",
              cmap=ListedColormap(ACTION_COLOUR), vmin=0, vmax=2, extent=[0, 23, 0, 1])
    ax.set_title(arch, fontsize=10)
    ax.set_xlabel("hour of day")
    ax.set_xticks(range(0, 24, 3))
axes[0].set_ylabel("fatigue $F$")
axes[1].legend([plt.Rectangle((0, 0), 1, 1, color=col) for col in ACTION_COLOUR],
               ["Hold", "Engage", "Incentive"], frameon=False, fontsize=8, loc="upper right")
fig.suptitle("G4   Myopic-optimal action by hour and fatigue", fontsize=11)
fig.tight_layout()
save(fig, "G4_ground_truth_policy")
plt.show()

---

## 6. Deep Q-Network (DQN) Agent Implementation

Defines `DQN_CONFIG`, `ReplayBuffer`, `QNetwork`, and the `DQNAgent` class.


In [ ]:
# --- Configuration ----------------------------------------------------------
DQN_CONFIG = dict(
    lr=1e-4,                  # Adam learning rate
    batch_size=64,            # Minibatch size
    max_grad_norm=0.5,        # Gradient clipping norm
    gamma=0.99,               # Discount factor
    epsilon_start=1.0,        # Initial exploration rate
    epsilon_end=0.05,         # Final exploration rate
    epsilon_decay_steps=50000,# Linear decay duration in steps
    buffer_size=10000,        # Replay buffer capacity
    min_replay_size=1000,     # Warmup steps before learning
    target_update_freq=500,   # Target network hard update frequency
    hidden_sizes=(64, 64),    # Q-network hidden layer dimensions
)

print("DQN_CONFIG:", ", ".join(f"{k}={v}" for k, v in DQN_CONFIG.items()))


In [ ]:
class ReplayBuffer:
    """Experience replay buffer storing (state, action, reward, next_state, done)."""

    def __init__(self, capacity, state_dim, seed=0):
        self.capacity = capacity
        self.rng = np.random.default_rng(seed)
        self.pos = 0
        self.size = 0

        self.states = np.zeros((capacity, state_dim), dtype=np.float32)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.next_states = np.zeros((capacity, state_dim), dtype=np.float32)
        self.dones = np.zeros(capacity, dtype=np.float32)

    def push(self, state, action, reward, next_state, done):
        self.states[self.pos] = state
        self.actions[self.pos] = action
        self.rewards[self.pos] = reward
        self.next_states[self.pos] = next_state
        self.dones[self.pos] = float(done)
        self.pos = (self.pos + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        idx = self.rng.integers(0, self.size, size=batch_size)
        return (
            torch.from_numpy(self.states[idx]),
            torch.from_numpy(self.actions[idx]),
            torch.from_numpy(self.rewards[idx]),
            torch.from_numpy(self.next_states[idx]),
            torch.from_numpy(self.dones[idx]),
        )

    def __len__(self):
        return self.size

print("ReplayBuffer defined.")

In [ ]:
class QNetwork(nn.Module):
    """Multi-Layer Perceptron Q-network."""

    def __init__(self, d_in, n_actions, hidden_sizes):
        super().__init__()
        self.net = self._mlp(d_in, hidden_sizes, n_actions, out_gain=0.01)

    @staticmethod
    def _mlp(d_in, hidden, d_out, out_gain):
        layers, prev = [], d_in
        for h in hidden:
            lin = nn.Linear(prev, h)
            nn.init.orthogonal_(lin.weight, gain=np.sqrt(2.0))
            nn.init.zeros_(lin.bias)
            layers += [lin, nn.Tanh()]
            prev = h
        head = nn.Linear(prev, d_out)
        nn.init.orthogonal_(head.weight, gain=out_gain)
        nn.init.zeros_(head.bias)
        layers.append(head)
        return nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

print("QNetwork defined.")

In [ ]:
class DQNAgent(Agent):
    """Deep Q-Network Agent (Mnih et al., 2015)."""

    def __init__(self, seed=0, label=None, **overrides):
        unknown = set(overrides) - set(DQN_CONFIG)
        if unknown:
            raise ValueError(f"unknown DQN hyperparameter(s): {sorted(unknown)}")
        self.cfg = {**DQN_CONFIG, **overrides}
        self._label = label

        self.rng = np.random.default_rng(seed)
        torch.manual_seed(seed)
        self.seed = seed

        self.d_in = agent_state_dim()
        self.online_net = QNetwork(self.d_in, N_ACTIONS, tuple(self.cfg["hidden_sizes"])).to(DEVICE)
        self.target_net = QNetwork(self.d_in, N_ACTIONS, tuple(self.cfg["hidden_sizes"])).to(DEVICE)
        self._sync_target()
        self.target_net.eval()

        self.opt = torch.optim.Adam(self.online_net.parameters(), lr=self.cfg["lr"])
        self.buffer = ReplayBuffer(self.cfg["buffer_size"], self.d_in, seed=seed)

        self.history = []
        self.n_updates = 0
        self.total_steps = 0

    def _features(self, state):
        x = np.asarray(state, dtype=np.float32)[:self.d_in].copy()
        x[IDX_HOUR] = x[IDX_HOUR] / 23.0
        x[IDX_DAY] = x[IDX_DAY] / 6.0
        return x

    def _epsilon(self):
        cfg = self.cfg
        frac = min(1.0, self.total_steps / max(1, cfg["epsilon_decay_steps"]))
        return cfg["epsilon_start"] + frac * (cfg["epsilon_end"] - cfg["epsilon_start"])

    def _sync_target(self):
        self.target_net.load_state_dict(self.online_net.state_dict())

    @property
    def name(self):
        return self._label or "DQN"

    def act(self, state, greedy=False):
        x = torch.from_numpy(self._features(state)).unsqueeze(0)
        with torch.no_grad():
            q_values = self.online_net(x).squeeze(0)

        if greedy:
            action = int(torch.argmax(q_values).item())
        else:
            eps = self._epsilon()
            if self.rng.random() < eps:
                action = int(self.rng.integers(N_ACTIONS))
            else:
                action = int(torch.argmax(q_values).item())

        return action, {"q_values": q_values.numpy().tolist()}

    def update(self, state, action, reward, next_state, done, aux):
        feat_s = self._features(state)
        feat_ns = self._features(next_state)

        self.buffer.push(feat_s, action, reward, feat_ns, done)
        self.total_steps += 1

        if len(self.buffer) < self.cfg["min_replay_size"]:
            return {}

        diag = self._optimise()

        if self.total_steps % self.cfg["target_update_freq"] == 0:
            self._sync_target()

        return diag

    def reset(self):
        pass

    def _optimise(self):
        cfg = self.cfg
        states, actions, rewards, next_states, dones = self.buffer.sample(cfg["batch_size"])

        q_all = self.online_net(states)
        q_values = q_all.gather(1, actions.unsqueeze(1).long()).squeeze(1)

        # Standard DQN Bellman target (Mnih et al., 2015)
        with torch.no_grad():
            next_q_target = self.target_net(next_states)
            max_next_q = next_q_target.max(dim=1).values
            target = rewards + cfg["gamma"] * max_next_q * (1.0 - dones)

        loss = nn.functional.mse_loss(q_values, target)

        self.opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.online_net.parameters(), cfg["max_grad_norm"])
        self.opt.step()

        self.n_updates += 1
        diag = {
            "loss": float(loss.detach()),
            "mean_q": float(q_values.mean().detach()),
            "max_q": float(q_values.max().detach()),
            "epsilon": self._epsilon(),
            "update": self.n_updates,
            "step": self.total_steps,
        }
        self.history.append(diag)
        return diag

    def state_dict_for_save(self):
        return {"online_net": self.online_net.state_dict(),
                "target_net": self.target_net.state_dict(),
                "cfg": dict(self.cfg),
                "d_in": self.d_in, "seed": self.seed,
                "n_updates": self.n_updates, "total_steps": self.total_steps}

_probe = DQNAgent(seed=0)
print("DQNAgent defined:", _probe)
print("parameters (online):", sum(p.numel() for p in _probe.online_net.parameters()))


---

## 7. Correctness Verification Tests

Suite of automated unit tests ensuring interface adherence, target computation logic, evaluation purity, known-optimum convergence, and seed reproducibility.

In [ ]:
# --- 7.1 Interface Conformance ----------------------------------------------
_env = CANEEnv(seed=0)
_s0, _ = _env.reset(seed=123)

assert isinstance(_probe, Agent)
assert _probe.name == "DQN"

_a, _aux = _probe.act(_s0, greedy=False)
assert isinstance(_a, int) and _a in (HOLD, ENGAGE, INCENTIVE)
assert isinstance(_aux, dict)

_greedy = {_probe.act(_s0, greedy=True)[0] for _ in range(100)}
assert len(_greedy) == 1

_s1, _r, _term, _trunc, _ = _env.step(_a)
_out = _probe.update(_s0, _a, _r, _s1, _term, _aux)
assert isinstance(_out, dict)

print("7.1 interface conformance PASS")

In [ ]:
# --- 7.2 Standard DQN Target Logic Test -----------------------------------
_test_agent = DQNAgent(seed=42, min_replay_size=10, target_update_freq=1000)
_test_env = CANEEnv(seed=999)

for _ in range(5):
    _s, _ = _test_env.reset(seed=None)
    for _ in range(168):
        _a, _aux = _test_agent.act(_s, greedy=False)
        _ns, _r, _term, _trunc, _ = _test_env.step(_a)
        _test_agent.update(_s, _a, _r, _ns, _term, _aux)
        _s = _ns
        if _term or _trunc:
            break

_test_state = torch.randn(1, _test_agent.d_in)
with torch.no_grad():
    _q_target = _test_agent.target_net(_test_state)
    _max_q_val = _q_target.max(dim=1).values[0]
    assert _max_q_val == torch.max(_q_target)

_online_params = list(_test_agent.online_net.parameters())
_target_params = list(_test_agent.target_net.parameters())
_any_different = any(not torch.equal(o, t) for o, t in zip(_online_params, _target_params))
assert _any_different

print("7.2 Standard DQN target logic PASS")


In [ ]:
# --- 7.3 Evaluation Purity Test ---------------------------------------------
_check = DQNAgent(seed=3)
_train_env = CANEEnv(seed=4321)
run_episodes(_check, _train_env, n_episodes=10, learn=True, greedy=False)
_buf_size_before = len(_check.buffer)

_before = [p.detach().clone() for p in _check.online_net.parameters()]
_updates_before = _check.n_updates

_m, _, _, _ = run_episodes(_check, CANEEnv(seed=4321), n_episodes=5, learn=False, greedy=True)
_after = [p.detach().clone() for p in _check.online_net.parameters()]

assert all(torch.equal(a, b) for a, b in zip(_before, _after))
assert _check.n_updates == _updates_before
assert len(_check.buffer) == _buf_size_before

print("7.3 evaluation purity PASS")

In [ ]:
# --- 7.4 Known-Optimum Learning Test --------------------------------------
def run_synthetic_dqn(agent, theta_star, n_episodes=80, ep_len=168, seed=0):
    rng = np.random.default_rng(seed)
    d = theta_star.shape[1]
    hits, rewards = [], []

    for _ in range(n_episodes):
        agent.reset()
        for t in range(ep_len):
            x = rng.normal(size=d).astype(np.float32)
            best = int(np.argmax(theta_star @ agent._features(x)))
            action, aux = agent.act(x, greedy=False)
            reward = 1.0 if action == best else 0.0
            x_next = rng.normal(size=d).astype(np.float32)
            agent.update(x, action, reward, x_next, False, aux)
            hits.append(action == best)
            rewards.append(reward)
    return np.array(hits), np.array(rewards)

_syn_agent = DQNAgent(seed=0, gamma=0.0, epsilon_decay_steps=5000)
_theta = np.random.default_rng(42).normal(size=(N_ACTIONS, _syn_agent.d_in))
_hits, _rew = run_synthetic_dqn(_syn_agent, _theta, n_episodes=80)

_first = _hits[:len(_hits) // 4].mean()
_last = _hits[-len(_hits) // 4:].mean()

assert _last > 0.65
assert _last > _first + 0.10
print("7.4 known-optimum learning PASS")

In [ ]:
# --- 7.5 Shared Environment Verification ----------------------------------
assert len(ARCHETYPES) == 5
assert CANE_CONFIG["steps_per_episode"] == 168
print("7.5 shared environment verification PASS")

In [ ]:
# --- 7.6 Seed Reproducibility Test -----------------------------------------
def _short_train_eval(seed, episodes=8):
    ag = DQNAgent(seed=seed)
    run_episodes(ag, CANEEnv(seed=1000 + seed), n_episodes=episodes, learn=True, greedy=False)
    m, _, _, _ = run_episodes(ag, CANEEnv(seed=7000 + seed), seeds=EVAL_SEEDS[:20], learn=False, greedy=True)
    return m["reward_mean"], ag

_r1, _a1 = _short_train_eval(0)
_r2, _a2 = _short_train_eval(0)
_r3, _a3 = _short_train_eval(1)

assert _r1 == _r2
assert all(torch.equal(p, q) for p, q in zip(_a1.online_net.parameters(), _a2.online_net.parameters()))

print("7.6 reproducibility PASS")

---

## 8. Agent Registry & Execution Driver

Registers baselines and the Deep Q-Network (DQN) agent, then executes evaluations across all user archetypes.


In [ ]:
AGENTS = {}

def register(name, factory, kind="learning"):
    if name in AGENTS:
        raise ValueError(f"{name} already registered")
    AGENTS[name] = {"factory": factory, "kind": kind}
    return name

register("Fixed-18:00", lambda s: FixedScheduleAgent(hour=18), kind="baseline")
register("Random",      lambda s: RandomAgent(seed=s),         kind="baseline")
register("DQN",        lambda s: DQNAgent(seed=s))

print(f"{len(AGENTS)} registered:", ", ".join(AGENTS))

In [ ]:
import time

RESULTS_ALL = {}
_t0 = time.time()

def run_all(archetype, names=None, n_seeds=None, train_episodes=TRAIN_EPISODES):
    out = {}
    for name in (names or list(AGENTS)):
        spec = AGENTS[name]
        if spec["kind"] == "baseline":
            r = evaluate_baseline(spec["factory"], archetype=archetype)
        else:
            r = train_and_evaluate(spec["factory"], n_seeds=n_seeds,
                                   train_episodes=train_episodes,
                                   archetype=archetype, collect_hours=True)
        r["agent"] = name
        out[name] = r
        print(f"  {archetype:16} {name:12} reward {r['reward_mean']:8.2f}  "
              f"ctr {r['ctr']:.3f}  sends {r['sends_per_episode']:6.2f}  "
              f"optout {r['optout_rate']:.3f}")
    return out

for arch in ARCHETYPES:
    RESULTS_ALL[arch] = run_all(arch)
RESULTS = {a: RESULTS_ALL[a] for a in FOCUS}

print(f"\n[{time.time() - _t0:.0f}s for full evaluation sweep]")

In [ ]:
from pathlib import Path

_resdir = Path("results")
_resdir.mkdir(exist_ok=True)

rows = []
for arch, res in RESULTS_ALL.items():
    for name, r in res.items():
        if "per_seed_rewards" in r:
            for si, sr in enumerate(r["per_seed_rewards"]):
                rows.append({"archetype": arch, "agent": name, "seed": si,
                             "reward_mean": sr, "ctr": r["ctr"],
                             "sends_per_episode": r["sends_per_episode"],
                             "optout_rate": r["optout_rate"]})
        else:
            rows.append({"archetype": arch, "agent": name, "seed": 0,
                         "reward_mean": r["reward_mean"], "ctr": r["ctr"],
                         "sends_per_episode": r["sends_per_episode"],
                         "optout_rate": r["optout_rate"]})

pd.DataFrame(rows).to_csv(_resdir / "dqn_results.csv", index=False)
print(f"Results saved to {_resdir / 'dqn_results.csv'}")

---

## 9. Learning Curves & Optimisation Diagnostics

Visualises policy learning over time (Q1) and loss/Q-value optimization metrics (Q2).


In [ ]:
dqn_trace_agent = DQNAgent(seed=0)
_snap_result = train_and_evaluate(
    lambda s: DQNAgent(seed=s), n_seeds=1,
    snapshot_every=50, snapshot_seed=0)

dqn_trace_agent = DQNAgent(seed=0)
_trace_env = CANEEnv(seed=1000)
run_episodes(dqn_trace_agent, _trace_env, n_episodes=TRAIN_EPISODES, learn=True, greedy=False)

In [ ]:
# --- Plot Q1: Learning Curve ---
snaps = _snap_result.get("snapshots", [])
if snaps:
    fig, axes = plt.subplots(1, len(FOCUS), figsize=(5.5 * len(FOCUS), 4.0), sharey=True, squeeze=False)
    for col, arch in enumerate(FOCUS):
        ax = axes[0, col]
        eps_list = [s[0] for s in snaps]
        rew_list = [s[1][arch]["reward"] for s in snaps]
        ax.plot(eps_list, rew_list, "o-", color=ARCH_COLOUR[arch], lw=1.8, markersize=4, label=arch)
        ax.axhline(0, color="#999999", lw=0.8, ls=":")
        ax.set_xlabel("training episodes")
        ax.set_ylabel("greedy reward / week" if col == 0 else "")
        ax.set_title(arch, fontsize=10)
        ax.grid(alpha=0.3)
    fig.suptitle("Q1   DQN Learning Curve", fontsize=11)
    fig.tight_layout()
    save(fig, "D1_ddqn_learning_curve")
    plt.show()

In [ ]:
# --- Plot Q2: Optimiser Diagnostics ---
hist = pd.DataFrame(dqn_trace_agent.history)

if len(hist) > 0:
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.0))
    for ax, (col, lab) in zip(axes, [("loss", "loss (MSE)"), ("mean_q", "mean Q"), ("max_q", "max Q"), ("epsilon", "epsilon")]):
        ax.plot(hist["update"], hist[col], lw=1.0, color="#2f6fb5", alpha=0.6)
        if len(hist) > 20:
            smoothed = hist[col].rolling(window=min(50, len(hist)//5), min_periods=1).mean()
            ax.plot(hist["update"], smoothed, lw=2.0, color="#c4453c")
        ax.set_xlabel("update")
        ax.set_title(lab, fontsize=9)
        ax.grid(alpha=0.3)
    fig.suptitle("Q2   DQN Optimiser Diagnostics", fontsize=11)
    fig.tight_layout()
    save(fig, "D2_ddqn_diagnostics")
    plt.show()

---

## 10. Decision Distribution & Policy Behaviour

Plots summary rewards by archetype (Q4) and hourly decision breakdown vs ground-truth policy (Q3).


In [ ]:
# --- Plot Q4: Reward by Archetype ---
names = list(AGENTS)
x = np.arange(len(ARCHETYPES))
width = 0.8 / len(names)

fig, ax = plt.subplots(figsize=(11, 3.6))
for i, name in enumerate(names):
    vals = [RESULTS_ALL[a][name]["reward_mean"] for a in ARCHETYPES]
    errs = [RESULTS_ALL[a][name]["reward_sem_std"] for a in ARCHETYPES]
    ax.bar(x + i * width - 0.4 + width / 2, vals, width * 0.92, yerr=errs,
           label=name, error_kw={"lw": 0.8, "ecolor": "#333333"})
ax.axhline(0.0, color="#333333", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(ARCHETYPES, fontsize=8)
ax.set_ylabel("greedy reward / week")
ax.legend(frameon=False, fontsize=8, ncol=len(names))
ax.set_title("D4   Reward by Agent and Archetype (+/- 1 SD)", fontsize=11)
fig.tight_layout()
save(fig, "D4_ddqn_reward_by_archetype")
plt.show()

In [ ]:
# --- Plot Q3: Decision Distribution ---
fig, axes = plt.subplots(2, len(FOCUS), figsize=(6 * len(FOCUS), 6),
                          gridspec_kw={"height_ratios": [6, 1]})
if len(FOCUS) == 1:
    axes = axes.reshape(2, 1)

for col, arch in enumerate(FOCUS):
    ax_main = axes[0, col]
    ax_gt = axes[1, col]

    ha = RESULTS_ALL[arch].get("DQN", {}).get("hour_actions", None)
    if ha is not None and ha.sum() > 0:
        totals = ha.sum(axis=1, keepdims=True)
        totals = np.where(totals == 0, 1, totals)
        fracs = ha / totals

        bottom = np.zeros(24)
        for a_idx, (a_name, colour) in enumerate(zip(["Hold", "Engage", "Incentive"], ACTION_COLOUR)):
            ax_main.bar(range(24), fracs[:, a_idx], bottom=bottom, width=0.9,
                        color=colour, label=a_name if col == 0 else None)
            bottom += fracs[:, a_idx]

    ax2 = ax_main.twinx()
    hours = np.arange(24)
    ax2.plot(hours, archetype_curve(arch), color="black", lw=1.5, ls="--", alpha=0.7)
    ax2.set_ylim(0, 1)
    ax2.set_ylabel("p0(h)", fontsize=8)

    ax_main.set_xlim(-0.5, 23.5)
    ax_main.set_ylim(0, 1)
    ax_main.set_xlabel("hour")
    ax_main.set_ylabel("action fraction" if col == 0 else "")
    ax_main.set_title(f"{arch} — DDQN", fontsize=10)

    gt = GROUND_TRUTH[arch]
    gt_mid = gt[len(gt) // 2]
    for h in range(24):
        ax_gt.barh(0, 1, left=h, color=ACTION_COLOUR[gt_mid[h]], edgecolor="none")
    ax_gt.set_xlim(-0.5, 23.5)
    ax_gt.set_yticks([])
    ax_gt.set_xlabel("hour")
    ax_gt.set_title("myopic optimal (F=0.5)", fontsize=8)

if len(FOCUS) > 0:
    axes[0, 0].legend(loc="upper left", fontsize=7, frameon=False)

fig.suptitle("D3   DDQN Decision Distribution by Hour", fontsize=11, y=1.02)
fig.tight_layout()
save(fig, "D3_ddqn_decision_distribution")
plt.show()

---

## 11. Hyperparameter Sensitivity Analysis

Performs a random search over learning rate, epsilon decay, update frequency, buffer size, and network dimensions (Q5).


In [ ]:
# --- Plot Q5: Hyperparameter Sensitivity ---
_search_rng = np.random.default_rng(2026)
_N_SEARCH = 20

_search_results = []
for i in range(_N_SEARCH):
    _lr = float(10 ** _search_rng.uniform(-5, -3))
    _eps_end = float(_search_rng.uniform(0.01, 0.15))
    _eps_decay = int(_search_rng.choice([20000, 50000, 80000, 100000]))
    _target_freq = int(_search_rng.choice([200, 500, 1000, 2000]))
    _buf_size = int(_search_rng.choice([5000, 10000, 20000]))
    _hidden = tuple(_search_rng.choice([(32, 32), (64, 64), (128, 128)]))

    try:
        r = train_and_evaluate(
            lambda s, _lr=_lr, _eps_end=_eps_end, _eps_decay=_eps_decay,
                   _target_freq=_target_freq, _buf_size=_buf_size,
                   _hidden=_hidden:
                DQNAgent(seed=s, lr=_lr, epsilon_end=_eps_end,
                          epsilon_decay_steps=_eps_decay,
                          target_update_freq=_target_freq,
                          buffer_size=_buf_size,
                          hidden_sizes=_hidden),
            n_seeds=2,
            archetype="OfficeWorker")
        _search_results.append({
            "lr": _lr, "eps_end": _eps_end, "eps_decay": _eps_decay,
            "target_freq": _target_freq, "buf_size": _buf_size,
            "hidden": str(_hidden), "reward": r["reward_mean"],
            "ctr": r["ctr"], "sends": r["sends_per_episode"],
            "optout": r["optout_rate"]})
    except Exception as ex:
        print(f"FAILED: {ex}")

_search_df = pd.DataFrame(_search_results)

if len(_search_df) > 0:
    _hp_cols = ["lr", "eps_end", "eps_decay", "target_freq", "buf_size"]
    fig, axes = plt.subplots(1, len(_hp_cols), figsize=(3.2 * len(_hp_cols), 3.0))

    for ax, col in zip(axes, _hp_cols):
        ax.scatter(_search_df[col], _search_df["reward"], s=30, alpha=0.7,
                   color="#2f6fb5", edgecolors="#333333", linewidth=0.5)
        ax.axhline(0, color="#999999", lw=0.8, ls=":")
        ax.set_xlabel(col, fontsize=8)
        ax.set_ylabel("reward" if col == _hp_cols[0] else "")
        ax.grid(alpha=0.3)
        if col == "lr":
            ax.set_xscale("log")

    fig.suptitle("D5   DDQN Random Search — OfficeWorker", fontsize=11)
    fig.tight_layout()
    save(fig, "D5_ddqn_random_search")
    plt.show()

---

## 12. Statistical Robustness & Model Persistence

Plots per-seed reward variance (Q6) and saves trained model weights.


In [ ]:
# --- Plot Q6: Robustness Boxplot ---
fig, axes = plt.subplots(1, len(FOCUS), figsize=(5.5 * len(FOCUS), 4.0), sharey=True, squeeze=False)

for col, arch in enumerate(FOCUS):
    ax = axes[0, col]
    agent_names = []
    agent_rewards = []
    for name in AGENTS:
        r = RESULTS_ALL[arch][name]
        if "per_seed_rewards" in r and len(r["per_seed_rewards"]) > 1:
            agent_names.append(name)
            agent_rewards.append(r["per_seed_rewards"])

    if agent_rewards:
        positions = range(len(agent_names))
        bp = ax.boxplot(agent_rewards, positions=list(positions), widths=0.5, patch_artist=True)
        for patch in bp["boxes"]:
            patch.set_facecolor("#2f6fb5")
            patch.set_alpha(0.5)
        ax.set_xticks(list(positions))
        ax.set_xticklabels(agent_names, fontsize=8)

    ax.axhline(0, color="#999999", lw=0.8, ls=":")
    ax.set_ylabel("greedy reward / week" if col == 0 else "")
    ax.set_title(arch, fontsize=10)
    ax.grid(alpha=0.3, axis="y")

fig.suptitle("D6   DDQN Robustness — Per-Seed Reward Distribution", fontsize=11)
fig.tight_layout()
save(fig, "D6_ddqn_robustness")
plt.show()

In [ ]:
# --- Model Checkpointing ---
_modeldir = Path("models")
_modeldir.mkdir(exist_ok=True)

for seed in range(N_SEEDS):
    agent = DQNAgent(seed=seed)
    train_env = CANEEnv(seed=1000 + seed)
    run_episodes(agent, train_env, n_episodes=TRAIN_EPISODES, learn=True, greedy=False)
    torch.save(agent.state_dict_for_save(), _modeldir / f"dqn_seed{seed}.pt")
    print(f"Saved model: dqn_seed{seed}.pt")

print("All models successfully saved.")

---

## Summary & Section Breakdown

This notebook is structured into 12 self-contained sections:

1. **Setup & Dependencies**: Environment configuration, imports, action constants, and feature index mapping.
2. **Agent Interface**: `Agent` abstract base class establishing the contract for all policies.
3. **Environment Dynamics**: `CANEEnv` implementation modeling receptive windows, fatigue accumulation, click probability, and churn hazard.
4. **Baselines & Evaluation Protocol**: Baseline policies (`FixedScheduleAgent`, `RandomAgent`) and evaluation harness (`run_episodes`, `train_and_evaluate`, `EVAL_SEEDS`).
5. **Sanity Checks & Environment EDA**: Unit verification of fatigue mechanics and initial environment visualization.
6. **Deep Q-Network Agent**: Core algorithm implementation containing `DQN_CONFIG`, `ReplayBuffer`, `QNetwork`, and Bellman error minimizing `DQNAgent`.
7. **Correctness Verification Tests**: Unit test suite (7.1 - 7.6) confirming interface compliance, standard DQN Bellman target computation, evaluation purity, synthetic problem convergence, and seed reproducibility.
8. **Registry & Execution Driver**: Multi-seed benchmark execution driver and CSV results exporter (`dqn_results.csv`).
9. **Learning Curves & Diagnostics**: Training progress curve (Q1) and optimization metrics trace (Q2: loss, Q-values, epsilon).
10. **Decision Distribution & Policy Behaviour**: Reward comparisons (Q4) and 24-hour action distribution vs. receptive windows (Q3).
11. **Hyperparameter Sensitivity Analysis**: Random search over hyperparameters (Q5).
12. **Statistical Robustness & Model Persistence**: Seed variance boxplots (Q6) and PyTorch checkpoint saving (`dqn_seed{0..4}.pt`).

### Figures Summary

- **Q1 (Learning Curve)**: Evaluates greedy weekly reward across probe training snapshots.
- **Q2 (Optimiser Diagnostics)**: Tracks MSE loss, mean $Q$-value, max $Q$-value, and linear $\epsilon$-decay per optimization update.
- **Q3 (Decision Distribution)**: Hourly stacked bar charts of policy action selection vs. baseline responsiveness and ground-truth policy.
- **Q4 (Reward by Archetype)**: Grouped bar chart comparing weekly greedy reward across Fixed-18:00, Random, and DQN.
- **Q5 (Random Search Scatter)**: Analyzes reward sensitivity across learning rate, $\epsilon$-decay, target update frequency, and buffer capacity.
- **Q6 (Robustness Boxplot)**: Displays reward variance across random seeds to verify statistical consistency.
- **G1-G4 (Shared Environment EDA)**: Archetype receptive windows, fatigue accumulation-recovery curves, click probability surfaces, and theoretical ground-truth policy maps.
